In [1]:
import os, gc, time
import pandas as pd, numpy as np, lightgbm as lgb
from sklearn.metrics import roc_auc_score

# Safely search for full.parquet
pq_files = [os.path.join(r, f) for r, _, fs in os.walk('/kaggle/input/notebooks/preranakamath21/nexusreturnguard-02-features') for f in fs if f == 'full.parquet']

if not pq_files:
    raise FileNotFoundError(
        "Could not find 'full.parquet' in '/kaggle/input'. "
        "Please check the right sidebar of your Kaggle notebook and make sure the correct dataset is added."
    )

PQ = pq_files[0]
full = pd.read_parquet(PQ)

ID_COLS = ['order_line_id','order_id','customer_id','product_id','store_id','supplier_code','banner_id']
NOT_FEATS = ID_COLS + ['order_ts','returned','is_train','month_idx','year']
FEATS = [c for c in full.columns if c not in NOT_FEATS]
CATS = [c for c in FEATS if str(full[c].dtype) == 'category']
print(full.shape, len(FEATS), len(CATS))     # must print (1880000, 104) 92 17

(1880000, 104) 92 17


In [2]:
params = dict(objective='binary', metric='auc', learning_rate=0.05, num_leaves=63,
              min_child_samples=200, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
              lambda_l2=5, max_bin=127, cat_smooth=20, min_data_per_group=200,
              n_jobs=-1, verbose=-1, seed=42)
fast = dict(params, learning_rate=0.1, num_leaves=31, max_bin=63)

P = ['prod_age', 'prod_prelaunch']; T = ['cpi', 'unemployment', 'fuel_price']; A = ['store_age', 'cust_tenure']
BASE = [c for c in FEATS if c != 'ord_dup_prod']

TE_KEYS = {'store_id': 50, 'product_id': 20, 'customer_id': 10, 'supplier_code': 50, 'banner_id': 50}
for k in TE_KEYS:
    full[k + '_cd'] = pd.factorize(full[k])[0].astype('int32')
train = full[full.is_train == 1]; test = full[full.is_train == 0]
fit = train[train.month_idx <= 18]; val = train[train.month_idx >= 19]

def add_te(df_fit, df_oth, keys=TE_KEYS):
    a, b = df_fit.copy(), df_oth.copy()
    prior = a['returned'].mean()
    for k, alpha in keys.items():
        c = k + '_cd'
        g = (a.groupby([c, 'month_idx'])['returned'].agg(s='sum', n='count')
               .reset_index().sort_values([c, 'month_idx']))
        g['ps'] = g.groupby(c)['s'].cumsum() - g['s']
        g['pn'] = g.groupby(c)['n'].cumsum() - g['n']
        a = a.merge(g[[c, 'month_idx', 'ps', 'pn']], on=[c, 'month_idx'], how='left')
        a[k + '_te'] = ((a['ps'] + prior * alpha) / (a['pn'] + alpha)).astype('float32')
        a = a.drop(columns=['ps', 'pn'])
        tot = df_fit.groupby(c)['returned'].agg(s='sum', n='count')
        s = b[c].map(tot['s']).fillna(0); n = b[c].map(tot['n']).fillna(0)
        b[k + '_te'] = ((s + prior * alpha) / (n + alpha)).astype('float32')
    return a, b

fit2, val2 = add_te(fit, val)
train2, test2 = add_te(train, test)
assert len(fit2) == len(fit) and len(val2) == len(val) and len(train2) == len(train) and len(test2) == len(test)

TE_COLS = [k + '_te' for k in TE_KEYS]
FEATS3 = [c for c in BASE if c not in (P + T + A)] + TE_COLS
FEATS4 = [c for c in FEATS3 if c != 'banner_id_te']
FEATS5 = [c for c in FEATS4 if c not in ('promo_num', 'promo_channel_code')] + ['banner_id_te']
SUB = '/kaggle/input/competitions/nexus-opti-ml-2026-operation-insight/sample_submission.csv'

def run_df(dfit, dval, feats, seed=42, rounds=1500, es=150):
    p = dict(fast, seed=seed); cats = [c for c in feats if c in CATS]
    dtr = lgb.Dataset(dfit[feats], dfit['returned'], categorical_feature=cats, free_raw_data=False)
    dva = lgb.Dataset(dval[feats], dval['returned'], categorical_feature=cats, reference=dtr, free_raw_data=False)
    m = lgb.train(p, dtr, rounds, valid_sets=[dva], valid_names=['val'],
                  callbacks=[lgb.early_stopping(es, verbose=False)])
    pv = m.predict(dval[feats], num_iteration=m.best_iteration)
    return m, m.best_iteration, roc_auc_score(dval['returned'], pv), pv

def month_auc(df, pcol, lo=19, hi=24):
    return [round(roc_auc_score(g['returned'], g[pcol]), 4) for m in range(lo, hi + 1) for g in [df[df.month_idx == m]]]
print(len(FEATS3), len(FEATS4), len(FEATS5))   # must print 89 88 87

89 88 87


## Task 12: Drift audit
**Why:** Val AUC 0.707, public LB 0.561. Something in test differs from everything val can show.
**Do:** (1) range check of growing counters, (2) adversarial val vs test on RAW features only, (3) signal strength per feature per period (promo is the control).
**Look for:** features whose strength falls like promo, counters with most test rows above the train range.

In [3]:
'''# 1. growing counters: how many test rows sit above the fit p99?
cnt = ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev', 'customer_id_freq',
       'product_id_freq', 'prod_freq_month', 'store_id_freq', 'order_value', 'items_in_order']
rows = []
for c in cnt:
    q = fit[c].quantile(0.99)
    rows.append((c, round(fit[c].mean(), 2), round(val[c].mean(), 2), round(test[c].mean(), 2),
                 round((val[c] > q).mean(), 3), round((test[c] > q).mean(), 3)))
print(pd.DataFrame(rows, columns=['feat', 'fit_mean', 'val_mean', 'test_mean', 'val>fit_p99', 'test>fit_p99']).to_string(index=False))

# 2. adversarial on RAW features (no TE, counters, freq, time, calendar)
SKIP = ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev', 'month', 'week_of_year',
        'fuel_surcharge', 'temperature_f', 'holiday_flag', 'prod_freq_month']
RAW = [c for c in FEATS5 if not c.endswith('_te') and not c.endswith('_freq') and c not in SKIP]
a = pd.concat([val2[val2.month_idx >= 22].sample(100_000, random_state=1).assign(adv=0),
               test2.sample(100_000, random_state=1).assign(adv=1)])
cut = np.random.RandomState(0).rand(len(a)) < 0.7
ac = [c for c in RAW if c in CATS]
ma = lgb.train(dict(objective='binary', metric='auc', learning_rate=0.1, num_leaves=31, verbose=-1, n_jobs=-1),
               lgb.Dataset(a.loc[cut, RAW], a.loc[cut, 'adv'], categorical_feature=ac), 150)
print('\nadversarial AUC raw features (0.5 = same):',
      round(roc_auc_score(a.loc[~cut, 'adv'], ma.predict(a.loc[~cut, RAW])), 4))
imp = pd.Series(ma.feature_importance('gain'), index=RAW)
print((imp / imp.sum() * 100).round(1).sort_values(ascending=False).head(8).to_string())

# 3. signal strength per period (weighted std of group return rates). promo = control
tr_ = train.copy()
tr_['per'] = pd.cut(tr_.month_idx, [-1, 12, 18, 21, 25], labels=['1-12', '13-18', '19-21', '22-25'])
for c in ['quantity', 'items_in_order', 'discount', 'unit_price', 'freight_cost', 'weight_kg']:
    tr_[c + '_b'] = pd.qcut(tr_[c], 10, duplicates='drop')
segs = ['promo_num', 'region_match', 'channel', 'customer_type', 'store_format', 'payment_method',
        'return_window_days', 'category', 'ship_mode', 'loyalty_tier', 'minute1', 'cust_region', 'store_region',
        'quantity_b', 'items_in_order_b', 'discount_b', 'unit_price_b', 'freight_cost_b', 'weight_kg_b']
def strength(df, c):
    g = df.groupby(c, observed=True)['returned'].agg(['mean', 'size'])
    return float(np.sqrt(((g['mean'] - df['returned'].mean()) ** 2 * g['size']).sum() / g['size'].sum()))
rows = [[c] + [round(strength(tr_[tr_.per == p], c), 4) for p in ['1-12', '13-18', '19-21', '22-25']] for c in segs]
st_ = pd.DataFrame(rows, columns=['feat', '1-12', '13-18', '19-21', '22-25'])
st_['late/early'] = (st_['22-25'] / st_['1-12']).round(2)
print('\n', st_.sort_values('late/early').to_string(index=False))'''

"# 1. growing counters: how many test rows sit above the fit p99?\ncnt = ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev', 'customer_id_freq',\n       'product_id_freq', 'prod_freq_month', 'store_id_freq', 'order_value', 'items_in_order']\nrows = []\nfor c in cnt:\n    q = fit[c].quantile(0.99)\n    rows.append((c, round(fit[c].mean(), 2), round(val[c].mean(), 2), round(test[c].mean(), 2),\n                 round((val[c] > q).mean(), 3), round((test[c] > q).mean(), 3)))\nprint(pd.DataFrame(rows, columns=['feat', 'fit_mean', 'val_mean', 'test_mean', 'val>fit_p99', 'test>fit_p99']).to_string(index=False))\n\n# 2. adversarial on RAW features (no TE, counters, freq, time, calendar)\nSKIP = ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev', 'month', 'week_of_year',\n        'fuel_surcharge', 'temperature_f', 'holiday_flag', 'prod_freq_month']\nRAW = [c for c in FEATS5 if not c.endswith('_te') and not c.endswith('_freq') and c not in SKIP]\na = pd.concat([v

## Task 13: Strip shifting features
**Why:** Adversarial val-vs-test AUC 0.793. Counters, freq, order-size, hour differ most. LB 0.561 vs val 0.707.
**Do:** Drop each group and all together. Same params, patience 150. Compare to val 0.7071 (noise 0.0003).
**Look for:** how much val drops. A cheap drop (under 0.01) = free robustness.

In [4]:
'''CNT = [c for c in ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev'] if c in FEATS5]
FRQ = [c for c in FEATS5 if c.endswith('_freq') or c == 'prod_freq_month']
ORD = [c for c in ['items_in_order', 'ord_total_qty', 'ord_max_price', 'ord_sum_line_value', 'ord_mean_price',
                   'ord_cat_repeat', 'ord_nunique_prod', 'ord_nunique_cat', 'ord_nunique_color',
                   'ord_n_same_cat', 'hour'] if c in FEATS5]
print('group sizes:', len(CNT), len(FRQ), len(ORD))

exps = {'drop counters': CNT, 'drop counters+freq': CNT + FRQ,
        'drop order-size+hour': ORD, 'drop ALL three': CNT + FRQ + ORD}
t0 = time.time()
for name, drop in exps.items():
    feats = [c for c in FEATS5 if c not in drop]
    m, it, a, pv = run_df(fit2, val2, feats)
    w = val2[['month_idx', 'returned']].copy(); w['p'] = pv
    print(f"{name:22} n_feats {len(feats):3} | iter {it:4} | val {a:.4f} | delta {a-0.7071:+.4f} | "
          f"per month {month_auc(w, 'p')} | {time.time()-t0:.0f}s")'''

'CNT = [c for c in [\'cust_order_seq\', \'cust_days_since_first\', \'cust_days_since_prev\'] if c in FEATS5]\nFRQ = [c for c in FEATS5 if c.endswith(\'_freq\') or c == \'prod_freq_month\']\nORD = [c for c in [\'items_in_order\', \'ord_total_qty\', \'ord_max_price\', \'ord_sum_line_value\', \'ord_mean_price\',\n                   \'ord_cat_repeat\', \'ord_nunique_prod\', \'ord_nunique_cat\', \'ord_nunique_color\',\n                   \'ord_n_same_cat\', \'hour\'] if c in FEATS5]\nprint(\'group sizes:\', len(CNT), len(FRQ), len(ORD))\n\nexps = {\'drop counters\': CNT, \'drop counters+freq\': CNT + FRQ,\n        \'drop order-size+hour\': ORD, \'drop ALL three\': CNT + FRQ + ORD}\nt0 = time.time()\nfor name, drop in exps.items():\n    feats = [c for c in FEATS5 if c not in drop]\n    m, it, a, pv = run_df(fit2, val2, feats)\n    w = val2[[\'month_idx\', \'returned\']].copy(); w[\'p\'] = pv\n    print(f"{name:22} n_feats {len(feats):3} | iter {it:4} | val {a:.4f} | delta {a-0.7071:+.4f} | "

## Task 14: Lean model (67 features)
**Why:** Adversarial AUC 0.793. Counters, freq encodings and order-size features shift most. Val cost to drop all: only -0.0051.
**Do:** Drop the 3 groups, retrain on all 24 months with 3 seeds averaged, write submission_lean.csv.
**Check:** pred mean ~0.18, 479,995 rows, no NaN. Compare LB vs Sub 1 (0.56084).

In [5]:
'''CNT = [c for c in ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev'] if c in FEATS5]
FRQ = [c for c in FEATS5 if c.endswith('_freq') or c == 'prod_freq_month']
ORD = [c for c in ['items_in_order', 'ord_total_qty', 'ord_max_price', 'ord_sum_line_value', 'ord_mean_price',
                   'ord_cat_repeat', 'ord_nunique_prod', 'ord_nunique_cat', 'ord_nunique_color',
                   'ord_n_same_cat', 'hour'] if c in FEATS5]
FEATS6 = [c for c in FEATS5 if c not in CNT + FRQ + ORD]
print(len(CNT), len(FRQ), len(ORD), '->', len(FEATS6), 'features')      # must print 3 6 11 -> 67

IT6 = 710                      # best iteration from Task 13 (drop ALL three)
n_rounds = int(IT6 * 1.15)     # 816, more data
cats = [c for c in FEATS6 if c in CATS]
dall = lgb.Dataset(train2[FEATS6], train2['returned'], categorical_feature=cats, free_raw_data=False)

preds, t0 = [], time.time()
for s in [42, 7, 11]:
    mf = lgb.train(dict(fast, seed=s), dall, n_rounds)
    preds.append(mf.predict(test2[FEATS6]))
    print('seed', s, 'done', f'{time.time()-t0:.0f}s')
print('seed corr:', np.corrcoef(preds[0], preds[1])[0, 1].round(4), np.corrcoef(preds[0], preds[2])[0, 1].round(4))
test_pred6 = np.mean(preds, axis=0)
np.save('/kaggle/working/test_lean.npy', test_pred6)
print('test pred mean', test_pred6.mean().round(4), '| min/max', test_pred6.min().round(3), test_pred6.max().round(3))

sub = pd.read_csv(SUB)
sub['returned'] = sub['order_line_id'].map(pd.Series(test_pred6, index=test2['order_line_id'].values)).values
assert len(sub) == 479_995 and sub['returned'].notna().all() and sub['returned'].between(0, 1).all()
sub.to_csv('/kaggle/working/submission_lean.csv', index=False)
print(sub.head(), '\nsaved')'''

"CNT = [c for c in ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev'] if c in FEATS5]\nFRQ = [c for c in FEATS5 if c.endswith('_freq') or c == 'prod_freq_month']\nORD = [c for c in ['items_in_order', 'ord_total_qty', 'ord_max_price', 'ord_sum_line_value', 'ord_mean_price',\n                   'ord_cat_repeat', 'ord_nunique_prod', 'ord_nunique_cat', 'ord_nunique_color',\n                   'ord_n_same_cat', 'hour'] if c in FEATS5]\nFEATS6 = [c for c in FEATS5 if c not in CNT + FRQ + ORD]\nprint(len(CNT), len(FRQ), len(ORD), '->', len(FEATS6), 'features')      # must print 3 6 11 -> 67\n\nIT6 = 710                      # best iteration from Task 13 (drop ALL three)\nn_rounds = int(IT6 * 1.15)     # 816, more data\ncats = [c for c in FEATS6 if c in CATS]\ndall = lgb.Dataset(train2[FEATS6], train2['returned'], categorical_feature=cats, free_raw_data=False)\n\npreds, t0 = [], time.time()\nfor s in [42, 7, 11]:\n    mf = lgb.train(dict(fast, seed=s), dall, n_rounds)\n    pre

## Task 15: Test regime check
**Why:** LB 0.56 vs val 0.70. Find what differs in months 25-31 and what the model can live without.
**Do:** (1) monthly share of minute1 / placeholder time, train and test, (2) mix shifts, (3) val AUC without TE and without minute flags.
**Look for:** a month where minute1 jumps or collapses, and the AUC cost of dropping TE.

In [6]:
'''# 1. monthly label-free shares, train + test (month_idx 25+ is test)
m = full.groupby('month_idx').agg(rows=('minute1', 'size'), minute1=('minute1', 'mean'),
                                  placeholder=('is_placeholder_time', 'mean'),
                                  region_match=('region_match', 'mean'))
m['ret_rate'] = full[full.is_train == 1].groupby('month_idx')['returned'].mean()
print(m.round(3).to_string())

# 2. category mix, fit / val / test
for c in ['channel', 'customer_type', 'store_format']:
    d = pd.concat([fit2[c].value_counts(normalize=True).rename('fit'),
                   val2[c].value_counts(normalize=True).rename('val'),
                   test2[c].value_counts(normalize=True).rename('test')], axis=1).fillna(0)
    print('\n', c); print(d.round(3).to_string())

# 3. what can the lean model live without?
NOTE = [c for c in FEATS6 if not c.endswith('_te')]
NOTM = [c for c in FEATS6 if c not in ('minute1', 'is_placeholder_time')]
t0 = time.time()
for name, feats in [('lean (67)', FEATS6), ('lean, no TE', NOTE), ('lean, no minute flags', NOTM)]:
    mm, it, a, pv = run_df(fit2, val2, feats)
    print(f"{name:22} n_feats {len(feats):3} | iter {it:4} | val {a:.4f} | {time.time()-t0:.0f}s")'''

'# 1. monthly label-free shares, train + test (month_idx 25+ is test)\nm = full.groupby(\'month_idx\').agg(rows=(\'minute1\', \'size\'), minute1=(\'minute1\', \'mean\'),\n                                  placeholder=(\'is_placeholder_time\', \'mean\'),\n                                  region_match=(\'region_match\', \'mean\'))\nm[\'ret_rate\'] = full[full.is_train == 1].groupby(\'month_idx\')[\'returned\'].mean()\nprint(m.round(3).to_string())\n\n# 2. category mix, fit / val / test\nfor c in [\'channel\', \'customer_type\', \'store_format\']:\n    d = pd.concat([fit2[c].value_counts(normalize=True).rename(\'fit\'),\n                   val2[c].value_counts(normalize=True).rename(\'val\'),\n                   test2[c].value_counts(normalize=True).rename(\'test\')], axis=1).fillna(0)\n    print(\'\n\', c); print(d.round(3).to_string())\n\n# 3. what can the lean model live without?\nNOTE = [c for c in FEATS6 if not c.endswith(\'_te\')]\nNOTM = [c for c in FEATS6 if c not in (\'minute1\'

## Task 16: Lean blend (LightGBM 3 seeds + CatBoost)
**Why:** Test behavior differs from val in ways we cannot see. Lean features + two algorithms = most robust we can build.
**Do:** 65 features (no promo, counters, freq, order-size, hour, minute flags). Train on all 24 months. Rank-average LGB and Cat.
**Check:** SMOKE run first (1 min). Then full run. 479,995 rows, no NaN.

In [7]:
'''from catboost import CatBoostClassifier, Pool
from scipy.stats import rankdata

SMOKE = False     # True = 1-minute plumbing test. Set False for the real run.

CNT = [c for c in ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev'] if c in FEATS5]
FRQ = [c for c in FEATS5 if c.endswith('_freq') or c == 'prod_freq_month']
ORD = [c for c in ['items_in_order', 'ord_total_qty', 'ord_max_price', 'ord_sum_line_value', 'ord_mean_price',
                   'ord_cat_repeat', 'ord_nunique_prod', 'ord_nunique_cat', 'ord_nunique_color',
                   'ord_n_same_cat', 'hour'] if c in FEATS5]
FEATS7 = [c for c in FEATS5 if c not in CNT + FRQ + ORD + ['minute1', 'is_placeholder_time']]
cats = [c for c in FEATS7 if c in CATS]
print(len(FEATS7), 'features |', len(cats), 'categorical')            # must print 65

dtrain = train2.sample(100_000, random_state=1) if SMOKE else train2
n_lgb, n_cb = (30, 30) if SMOKE else (940, 700)                      # 819 x1.15 and 605 x1.15 (+~)

# 1. LightGBM, 3 seeds
dall = lgb.Dataset(dtrain[FEATS7], dtrain['returned'], categorical_feature=cats, free_raw_data=False)
t0, pl = time.time(), []
for s in [42, 7, 11]:
    pl.append(lgb.train(dict(fast, seed=s), dall, n_lgb).predict(test2[FEATS7]))
    print('lgb seed', s, f'{time.time()-t0:.0f}s')
p_lgb = np.mean(pl, axis=0)

# 2. CatBoost (no eval set, fixed iterations)
def to_cb(df):
    x = df[FEATS7].copy()
    for c in cats: x[c] = x[c].astype(str)
    return x
cb = CatBoostClassifier(iterations=n_cb, learning_rate=0.1, depth=7, border_count=64, l2_leaf_reg=5,
                        thread_count=-1, random_seed=42, verbose=100)
cb.fit(Pool(to_cb(dtrain), dtrain['returned'], cat_features=cats))
p_cb = cb.predict_proba(Pool(to_cb(test2), cat_features=cats))[:, 1]
print(f'cat done {time.time()-t0:.0f}s')

# 3. rank blend + submission
r = lambda x: rankdata(x) / len(x)
blend = 0.5 * r(p_lgb) + 0.5 * r(p_cb)
print('lgb mean', p_lgb.mean().round(4), '| cat mean', p_cb.mean().round(4),
      '| corr lgb/cat', np.corrcoef(p_lgb, p_cb)[0, 1].round(4))
tag = 'SMOKE' if SMOKE else 'blend'
np.save(f'/kaggle/working/test_{tag}.npy', blend)
sub = pd.read_csv(SUB)
sub['returned'] = sub['order_line_id'].map(pd.Series(blend, index=test2['order_line_id'].values)).values
assert len(sub) == 479_995 and sub['returned'].notna().all() and sub['returned'].between(0, 1).all()
sub.to_csv(f'/kaggle/working/submission_{tag}.csv', index=False)
print(sub.head(), '\nsaved', tag)'''

"from catboost import CatBoostClassifier, Pool\nfrom scipy.stats import rankdata\n\nSMOKE = False     # True = 1-minute plumbing test. Set False for the real run.\n\nCNT = [c for c in ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev'] if c in FEATS5]\nFRQ = [c for c in FEATS5 if c.endswith('_freq') or c == 'prod_freq_month']\nORD = [c for c in ['items_in_order', 'ord_total_qty', 'ord_max_price', 'ord_sum_line_value', 'ord_mean_price',\n                   'ord_cat_repeat', 'ord_nunique_prod', 'ord_nunique_cat', 'ord_nunique_color',\n                   'ord_n_same_cat', 'hour'] if c in FEATS5]\nFEATS7 = [c for c in FEATS5 if c not in CNT + FRQ + ORD + ['minute1', 'is_placeholder_time']]\ncats = [c for c in FEATS7 if c in CATS]\nprint(len(FEATS7), 'features |', len(cats), 'categorical')            # must print 65\n\ndtrain = train2.sample(100_000, random_state=1) if SMOKE else train2\nn_lgb, n_cb = (30, 30) if SMOKE else (940, 700)                      # 819 x1.15 and 605

## Task 17: No target encoding
**Why:** LB plateau at 0.564 for 3 different models. Common part: target encodings. Test whether id history carries into months 25-30.
**Do:** 60 features (lean, no promo, no minute flags, no TE). 3 seeds, all 24 months, LightGBM only.
**Check:** SMOKE first. Then compare LB to 0.56367. If clearly higher, TE was the culprit.

In [8]:
SMOKE = False     # True = 1-minute plumbing test, then set False

CNT = [c for c in ['cust_order_seq', 'cust_days_since_first', 'cust_days_since_prev'] if c in FEATS5]
FRQ = [c for c in FEATS5 if c.endswith('_freq') or c == 'prod_freq_month']
ORD = [c for c in ['items_in_order', 'ord_total_qty', 'ord_max_price', 'ord_sum_line_value', 'ord_mean_price',
                   'ord_cat_repeat', 'ord_nunique_prod', 'ord_nunique_cat', 'ord_nunique_color',
                   'ord_n_same_cat', 'hour'] if c in FEATS5]
FEATS8 = [c for c in FEATS5 if c not in CNT + FRQ + ORD + TE_COLS + ['minute1', 'is_placeholder_time']]
cats = [c for c in FEATS8 if c in CATS]
print(len(FEATS8), 'features |', len(cats), 'categorical | TE left:', [c for c in FEATS8 if c.endswith('_te')])

dtrain = train2.sample(100_000, random_state=1) if SMOKE else train2
n_rounds = 30 if SMOKE else 1000        # no-TE best iter ~905 x 1.15
dall = lgb.Dataset(dtrain[FEATS8], dtrain['returned'], categorical_feature=cats, free_raw_data=False)

t0, preds = time.time(), []
for s in [42, 7, 11]:
    preds.append(lgb.train(dict(fast, seed=s), dall, n_rounds).predict(test2[FEATS8]))
    print('seed', s, f'{time.time()-t0:.0f}s')
p = np.mean(preds, axis=0)
print('pred mean', p.mean().round(4), '| min/max', p.min().round(3), p.max().round(3),
      '| seed corr', np.corrcoef(preds[0], preds[1])[0, 1].round(4))

tag = 'SMOKE_noTE' if SMOKE else 'noTE'
np.save(f'/kaggle/working/test_{tag}.npy', p)
sub = pd.read_csv(SUB)
sub['returned'] = sub['order_line_id'].map(pd.Series(p, index=test2['order_line_id'].values)).values
assert len(sub) == 479_995 and sub['returned'].notna().all() and sub['returned'].between(0, 1).all()
sub.to_csv(f'/kaggle/working/submission_{tag}.csv', index=False)
print(sub.head(), '\nsaved', tag)

60 features | 16 categorical | TE left: []
seed 42 140s
seed 7 278s
seed 11 416s
pred mean 0.1838 | min/max 0.015 0.855 | seed corr 0.9612
     order_line_id  returned
0  L85d608999fa1ee  0.133606
1  L86272cb9543018  0.394763
2  L66b5669bdace5b  0.107113
3  L2c5886124612de  0.126866
4  L56fe7aa5ac5dd8  0.131420 
saved noTE
